# Downscale DAMIP temperature data
 
> This notebook is part of the scientific analysis for the paper **"Heat-related mortality attributable to human-induced climate change: findings from the 100 Million Brazilian Cohort" (2025)**.  
>
> It contains code to downscale Detection and Attribution Model Intercomparison Project (DAMIP) model data to 0.5 degrees using the Inter-Sectorial Impact Model Intercomparison Project 3 (ISIMIP3) statistical downscaling method.  
>  
> This notebook uses [code](https://zenodo.org/records/7151476) developed for the bias adjustment and statistical downscaling of ISIMIP3. 
>
>---
>
>### License
>
>This notebook is licensed under the [Creative Commons Attribution 4.0 International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).  
>You are free to share and adapt the material with appropriate credit.
>
>---
>
>### Authors
>
>- **Ruby Lieber** – School of Geographical Sciences, University of Bristol 
>- **Emily Vosper** – School of Geographical Sciences, University of Bristol 
>
>---
>
>### Dependencies
>
>This notebook was developed with **Python 3.13** and using **Jupyter Notebook**. Functions can be found in the GitHub repository. 
>
>Required packages (see also `pixi.toml`):  
>```bash
>numpy
>xarray
>xesmf
>```

#### Imports

In [1]:
import os
import re
import xarray as xr
import numpy as np
import xesmf as xe
from glob import glob
from collections import defaultdict
import functions as func
import subprocess
import tempfile

#### DAMIP data 
Temperature (tas) data for three DAMIP models was downloaded from ESGF servers. This cell reads in the hist-nat model run. Uncomment alternative path to run on historical run. File paths will need to be changed.

In [ ]:
base_path = "/bias_correct_damip/"
pattern = f"{base_path}tas_*_hist-nat_*_bias_corrected_2000_2018.nc"
# pattern = f"{base_path}tas_*_historical_*_bias_corrected_2000_2018.nc"

temp_corrected = {}

for path in glob(pattern):
    match = re.match(r".*tas_(.+)_hist-nat_(r\d+i\d+p\d+f\d+)_bias_corrected_2000_2018\.nc", path)
    # match = re.match(r".*tas_(.+)_historical_(r\d+i\d+p\d+f\d+)_bias_corrected_2000_2018\.nc", path)
    if not match:
        print(f"Skipping unmatched file: {path}")
        continue

    model, ensemble = match.groups()
    ds = xr.open_dataset(path)
    ds = ds.assign_coords(ensemble=ensemble)

    if model not in temp_corrected:
        temp_corrected[model] = []

    temp_corrected[model].append(ds)

corrected = {}
for model, datasets_list in temp_corrected.items():
    corrected[model] = xr.concat(datasets_list, dim="ensemble", coords="all", join="override")

#### Observational data 
Daily average temperature was obtained from the Brazilian Daily Weather Gridded Data (BR-DWGD) produced by Xavier et al. (2022). NetCDF downloads are available [here](https://sites.google.com/site/alexandrecandidoxavierufes/brazilian-daily-weather-gridded-data?authuser=0). File paths will need to be changed. 

In [ ]:
# Read in obs interp and regrid to 0.5°
obs = xr.open_dataarray("obs_padded_interp_01x01_2000_2018.nc")

# Create lat/lon arrays with 0.5° spacing
lat_new = np.arange(-35, 6.5+0.5, 0.5)   
lon_new = np.arange(-75, -33.5+0.5, 0.5)

target_grid = xr.Dataset({
    "lat": (["lat"], lat_new),
    "lon": (["lon"], lon_new),
})

regridder = xe.Regridder(
    obs, 
    target_grid, 
    method="bilinear", 
    unmapped_to_nan=True
)

obs_05x05 = regridder(obs)

#### Preprocessing function
This function ensures that the resolution of the observations and the resolution of the model is compatible with the downscaling script.

In [4]:
def prepare_bias_corrected_and_obs(
    bias_corrected: xr.Dataset, 
    obs_05x05: xr.Dataset,
    obs_resolution: float = 0.5
) -> tuple[xr.Dataset, xr.Dataset]:
    """
    Aligns bias-corrected model output with observed data:
    - Trims lat and lon so that len(obs) / len(model) = integer (scale factor)
    - Ensures spatial grid centers align
    - Standardizes variable name and dimension order
    - Converts calendar to Proleptic Gregorian

    Parameters
    ----------
    bias_corrected : xr.Dataset
        Bias-corrected model output with dimensions (time, lat, lon).
    obs_05x05 : xr.Dataset
        Observational dataset to match with model.
    obs_resolution : float, optional
        Spatial resolution of observational data (default is 0.5).

    Returns
    -------
    tuple[xr.Dataset, xr.Dataset]
        Tuple of (bias_corrected_trimmed, obs_05x05_trimmed).
    """

    # Trim model and obs to have compatible lat lon grids 
    bias_trim, obs_trim = func.trim_model_and_obs_to_match(
        bias_corrected, obs_05x05, obs_resolution
    )

    # Align grid centers
    lat_adjust = func.compare_grids(bias_trim, obs_trim, "lat")
    lon_adjust = func.compare_grids(bias_trim, obs_trim, "lon")
    bias_trim = bias_trim.assign_coords(
        lat=bias_trim.lat - lat_adjust,
        lon=bias_trim.lon - lon_adjust
    )

    # Rename and reorder dimensions
    obs_trim.name = "tas"
    obs_trim = obs_trim.transpose("lat", "lon", "time").chunk({
        "time": -1, "lat": 'auto', "lon": 'auto'
    })
    obs_trim = obs_trim.convert_calendar("proleptic_gregorian", use_cftime=True)

    bias_trim = bias_trim.transpose("lat", "lon", "time").chunk({
        "time": -1, "lat": 'auto', "lon": 'auto'
    })
    bias_trim = bias_trim.convert_calendar("proleptic_gregorian", use_cftime=True)

    return bias_trim, obs_trim

#### Apply downscaling
Downscaling is performed on the hist-nat model run (uncomment alternative path to run on historical). The script "statistical_downscaling.py" is run using a subprocess within the jupyter notebook. It can be run through the command line if preprocessed files are saved. 

In [ ]:
# Select model(s) if need to do one at a time
# corrected = {key: corrected[key] for key in ['CanESM5', 'GFDL-CM4']}

In [ ]:
for model, ds in corrected.items():
    for ens in ds.ensemble.values:
        try:
            bias_corrected = ds.sel(ensemble=ens).tas
            bias_trim, obs_trim = prepare_bias_corrected_and_obs(bias_corrected, obs_05x05)
            # Drop ensemble coord
            bias_trim = bias_trim.reset_coords('ensemble', drop=True)
        except Exception as e:
            print(f"Skipping {model} {ens}: error during preparation -> {e}")
            continue

        with tempfile.TemporaryDirectory() as tmpdir:
            obs_path = f"{tmpdir}/tas_obs.nc"
            model_path = f"{tmpdir}/tas_model.nc"
            downscaled_path = f"/downscale_damip_05/tas_{model}_hist-nat_{ens}_downscaled_05_2000_2018.nc"
            # downscaled_path = f"/downscale_damip_05/tas_{model}_historical_{ens}_downscaled_05_2000_2018.nc"

            # Save prepared datasets
            obs_trim.to_netcdf(obs_path)
            bias_trim.to_netcdf(model_path)

            cmd = [
                "python", "-u", "statistical_downscaling.py",
                "--n-processes", "12",
                "--randomization-seed", "0",
                "-v", "tas",
                "-o", obs_path,
                "-s", model_path,
                "-f", downscaled_path,
            ]

            print(f"Running downscaling for {model} {ens}...")
            result = subprocess.run(cmd, capture_output=True, text=True)
            # print(result.stdout)
            if result.returncode != 0:
                print(f"Error in {model} {ens}:", result.stderr)